In [1]:
%load_ext autoreload
%autoreload 2

import numpy as np
import time
import pickle
import matplotlib.pyplot as plt
from scipy.stats import norm

from e_1_run_cvae import * 
from e_2_CVAE import *
from f_compare_bn import *

# global var
S0 = 1.0
K = 1.0
r = 0.03
sigma = np.sqrt(0.05)
T = 1.5
BS_eta = (S0, K, r, sigma, T)
# S0, K, r, kappa, theta, xi, rho, Y0, T = Hes_eta
Hes_eta = (S0, K, r, 2, 0.05, 0.5, -0.7, 0.05, T)

B = 0.8 # down-and-out must B < S0 and B < K
opt_type = 'call' # call or put
barr_type = 'van' # van or barr
model_type = 'bs' # hes or bs

if not((B < S0) & (B < K)):
    raise ValueError("down-and-out : B should be smaller than S0 and K")

if not(opt_type == 'call' or  opt_type == 'put'):
    raise ValueError("option_type must be 'call' or 'put'")

if not(barr_type == 'van' or  barr_type == 'barr'):
    raise ValueError("barr_type must be 'van' or 'barr'")

if not(model_type == 'hes' or  model_type == 'bs'):
    raise ValueError("model_type must be 'hes' or 'bs'")

# cvae training settings
if model_type == 'hes':
    if barr_type == 'barr':
        if opt_type == 'call':
            bench_price = 0.115733
        else: # put
            bench_price = 0.005170
    else: # van
        if opt_type == 'call':
            bench_price = 0.124491
        else: # put
            bench_price = 0.080488

elif model_type == 'bs':
    if barr_type == 'barr':
        if opt_type == 'call':
            bench_price = 0.123493
        else: # put
            bench_price = 0.009535
    else: # van
        if opt_type == 'call':
            bench_price = 0.129944
        else: # put
            bench_price = 0.085942

/home/enjongoopee/.local/lib/python3.12/site-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "


# =====================
# training
# =====================

In [ ]:
dim_z       = 8 # 12
hidden_dims = [512, 512, 256] # [128, 128, 64], [256, 256, 128], [512, 512, 256]
batch_size  = 8192 # 1024, 2048, 4096, 8192
n_epochs    = 5 # loss 수렴할 때까지 
lr          = 1e-3 # 3e-4, 5e-4
beta        = 1.0
use_bn = False
save_path = f"cvae_{model_type}_{barr_type}_{dim_z}_{hidden_dims[0]}\
    _{batch_size}_epoch1_NBN.pt"
load_path = None
resume_path = None


n_samples = 10000 # n_samples= 1k, 10k, 100k
if n_samples % 2 != 0:
    raise ValueError("n_samples should be an even number for antithetic sampling")

if model_type == 'hes':
    test_etas = [0.03, 2.0,  0.05, 0.5, -0.7, 0.05, 1.5]
    eta_keys  = ['r', 'lambda', 'v_bar', 'xi', 'rho', 'Y0', 'T']
else: # model_type = 'bs'
    test_etas = [r, sigma, T]
    eta_keys  = ['r', 'sigma', 'T']

time1 = time.time()
cvae, loss_history, eta_min, eta_max = train_chunk(
    model_type, barr_type, dim_z, hidden_dims, batch_size, n_epochs, lr, beta, use_bn,
    save_path, load_path, resume_path
)
time2 = time.time()
print(f"Training time: {time2 - time1:.6f}s")

In [ ]:
if barr_type == 'barr':
    compare_prices(cvae, B, K, eta_min, eta_max, test_etas, eta_keys, bench_price, 
                opt_type, barr_type, n_samples) 
elif barr_type == 'van':
    compare_prices(cvae, None, K, eta_min, eta_max, test_etas, eta_keys, bench_price, 
                opt_type, barr_type, n_samples)

epochs = range(1, len(loss_history["total_loss"]) + 1)
plt.plot(epochs, loss_history["total_loss"], label="Total", color="green", linewidth=2)
plt.plot(epochs, loss_history["recon_loss"], label="Recon", color="blue", linestyle="--", linewidth=2)
plt.plot(epochs, loss_history["KL_loss"], label="KL", color="orange", linewidth=2)
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.show()

# =======================================
# compare distribution of X_T and M_T
# ========================================

In [ ]:
# real X, M
real_x, real_m = load_real_xt_mt_from_chunks(
    chunk_dir="/mnt/d/bs_chunks_correction/",
    num_chunks=10,
)

statics_result("Real X_T", real_x)
if len(real_m) != 0:
    statics_result("Real M_T", real_m)

plot_1d("Real X_T", real_x, bins=100)
if len(real_m) != 0:
    plot_1d("Real M_T", real_m, bins=100)
    plot_2d_density(
        real_x, real_m,
        title="Real MC",
        xlabel="X_T",
        ylabel="M_T",
        bins=100,
    )

In [ ]:
# use BN
x_bn, m_bn, ckpt_bn = load_cvae_xt_mt(
    save_path=save_path_bn,
    test_etas=test_etas,
    n_samples=len(real_x),
)

statics_result("CVAE BN X_T", x_bn)
if len(m_bn) != 0:
    statics_result("CVAE BN M_T", m_bn)


plot_1d("CVAE BN X_T", x_bn, bins=100)
if len(m_bn) != 0:
    plot_1d("CVAE BN M_T", m_bn, bins=100)
    plot_2d_density(
        x_bn, m_bn,
        title="CVAE BN",
        xlabel="X_T",
        ylabel="M_T",
        bins=100,
    )


In [ ]:
# not use BN
x_no_bn, m_no_bn, ckpt_no_bn = load_cvae_xt_mt(
    save_path=save_path_no_bn,
    test_etas=test_etas,
    n_samples=len(real_x),
)

statics_result("CVAE no BN X_T", x_no_bn)
if len(m_no_bn) != 0:
    statics_result("CVAE no BN M_T", m_no_bn)

plot_1d("CVAE no BN X_T", x_no_bn, bins=100)
if len(m_no_bn) != 0:
    plot_1d("CVAE no BN M_T", m_no_bn, bins=100)
    plot_2d_density(
        x_no_bn, m_no_bn,
        title="CVAE no BN",
        xlabel="X_T",
        ylabel="M_T",
        bins=100,
    )


In [ ]:
# vanilla X_t
plot_three_distributions(real_x, x_no_bn, x_bn, name="X_T", bins=100)

real_x, real_m = load_real_xt_mt_from_chunks(chunk_dir, num_chunks=10)
x_no_bn, m_no_bn, _ = load_cvae_xt_mt(save_path_no_bn, test_etas, n_samples=len(real_x))
x_bn, m_bn, _ = load_cvae_xt_mt(save_path_bn, test_etas, n_samples=len(real_x))

real_price = vanilla_price_from_xt(real_x, K, r, T, opt_type)
no_bn_price = vanilla_price_from_xt(x_no_bn, K, r, T, opt_type)
bn_price = vanilla_price_from_xt(x_bn, K, r, T, opt_type)

print("Real vanilla price    :", real_price)
print("No BN price   :", no_bn_price, f"error={price_error(no_bn_price, real_price):+.2f}%")
print("BN price      :", bn_price, f"error={price_error(bn_price, real_price):+.2f}%")

In [ ]:
# barrier M_t
plot_three_distributions(real_m, m_no_bn, m_bn, name="M_T", bins=100)

real_x, real_m = load_real_xt_mt_from_chunks(chunk_dir, num_chunks=10)
x_no_bn, m_no_bn, _ = load_cvae_xt_mt(save_path_no_bn, test_etas, n_samples=len(real_x))
x_bn, m_bn, _ = load_cvae_xt_mt(save_path_bn, test_etas, n_samples=len(real_x))

real_price = barrier_price_from_xt_mt(real_x, real_m, K, r, T, opt_type)
no_bn_price = barrier_price_from_xt_mt(x_no_bn, m_no_bn, K, r, T, opt_type)
bn_price = barrier_price_from_xt_mt(x_bn, m_bn, K, r, T, opt_type)

print("Real barrier price    :", real_price)
print("No BN price   :", no_bn_price, f"error={price_error(no_bn_price, real_price):+.2f}%")
print("BN price      :", bn_price, f"error={price_error(bn_price, real_price):+.2f}%")

# =========================
# inference
# ========================

In [ ]:
save_path = "cvae_bs_van_8_512_8192_old.pt"
trained_model = torch.load(save_path)

cvae = CVAE(
    dim_x       = trained_model['dim_x'],
    dim_eta     = trained_model['dim_eta'],
    dim_z       = trained_model['dim_z'],
    hidden_dims = trained_model['hidden_dims']
)
cvae.load_state_dict(trained_model['model_state'])
cvae.eval()

eta_min = trained_model['eta_min']
eta_max = trained_model['eta_max']

# η 정규화 후 가격 산정
eta_raw    = np.array(test_etas, dtype=np.float32)  
eta_scaled = (eta_raw - eta_min) / (eta_max - eta_min + 1e-8)
eta_t      = torch.tensor(eta_scaled, dtype=torch.float32)

n_list    = [1000, 10000, 100000]
n_repeats = 50
results   = {n: [] for n in n_list}

for n in n_list:
    for _ in range(n_repeats):
        if barr_type == 'barr':
            cvae_price = cvae.price_barrier(eta_t, B, K, r, T, opt_type, n)
        else:
            cvae_price = cvae.price_vanilla(eta_t, K, r, T, opt_type, n)
        results[n].append(cvae_price)

with open(f"cvae_{model_type}_{barr_type}_{opt_type}_results.pkl", "wb") as f:
    pickle.dump(results, f)

FileNotFoundError: [Errno 2] No such file or directory: 'cvae_bs_van_8_512_8192_old.pt'

In [ ]:
# 공용 그래프
n_list    = [1000, 10000, 100000]
method = 'cvae' # 'cvae', 'mc'
with open(f'{method}_{model_type}_{barr_type}_{opt_type}_results.pkl', 'rb') as f:
    results = pickle.load(f)
    
means  = [np.mean(results[n]) for n in n_list]
stds   = [np.std(results[n])  for n in n_list]
ci     = [1.96 * s for s in stds]

plt.figure(figsize=(7, 4))
plt.axhline(bench_price, color='gray', linewidth=1.5, label='FDM')
plt.errorbar(range(len(n_list)), means, yerr=ci,
             fmt='o-', color='steelblue', capsize=5, label=f'{method} (95% CI)')
plt.xticks(range(len(n_list)), ['1K', '10K', '100K'])
plt.xlabel('Number of simulations')
plt.ylabel('Option price')
plt.title('convergence')
plt.legend()
plt.tight_layout()